In [ ]:
import kagglehub
import os

# ⚠️ Better practice: use secrets, but keeping your example intact
os.environ["KAGGLE_USERNAME"] = "ravikshdikola"
os.environ["KAGGLE_KEY"] = "apply token here"

# 👇 Force kagglehub cache to Colab filesystem
#os.environ["KAGGLEHUB_CACHE_DIR"] = "/content/kagglehub"

# # Download dataset
path = kagglehub.dataset_download("ravikshdikola/dpc2-exp2")

print("Dataset downloaded to:", path)


100%|██████████| 10.2G/10.2G [08:03<00:00, 22.6MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/ravikshdikola/dpc2-exp2/versions/1


In [ ]:
!cp -r /root/.cache/kagglehub/datasets/ravikshdikola/dpc2-exp2/versions/1  /content/model3/

In [ ]:
import kagglehub
import os

# ⚠️ Better practice: use secrets, but keeping your example intact
os.environ["KAGGLE_USERNAME"] = "ravikshdikola"
os.environ["KAGGLE_KEY"] = "apply token here"

# 👇 Force kagglehub cache to Colab filesystem
#os.environ["KAGGLEHUB_CACHE_DIR"] = "/content/kagglehub"

# # Download dataset
path = kagglehub.dataset_download("ravikshdikola/madlab-akkadian-retrained-1st-stage-3mix-new")

print("Dataset downloaded to:", path)


100%|██████████| 10.2G/10.2G [08:01<00:00, 22.7MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/ravikshdikola/madlab-akkadian-retrained-1st-stage-3mix-new/versions/1


In [ ]:
!cp -r /root/.cache/kagglehub/datasets/ravikshdikola/madlab-akkadian-retrained-1st-stage-3mix-new/versions/1  /content/model2/

In [ ]:
import kagglehub
import os

# ⚠️ Better practice: use secrets, but keeping your example intact
os.environ["KAGGLE_USERNAME"] = "ravikshdikola"
os.environ["KAGGLE_KEY"] = "apply token here"

# 👇 Force kagglehub cache to Colab filesystem
#os.environ["KAGGLEHUB_CACHE_DIR"] = "/content/kagglehub"

# # Download dataset
path = kagglehub.dataset_download("ravikshdikola/madlab-akkadian-retrained-1st-stage-ciphermix-2ep")

print("Dataset downloaded to:", path)


100%|██████████| 10.2G/10.2G [08:07<00:00, 22.4MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/ravikshdikola/madlab-akkadian-retrained-1st-stage-ciphermix-2ep/versions/1


In [ ]:
!cp -r /root/.cache/kagglehub/datasets/ravikshdikola/madlab-akkadian-retrained-1st-stage-ciphermix-2ep/versions/1  /content/model1/

In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00


In [ ]:
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.6 MB/s eta 0:00:00


In [ ]:
!pip install transformers==4.57.6


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 202.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 73.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.5.0
    Uninstalling huggingface_hub-1.5.0:
      Successfully uninstalled huggingface_hub-1.5.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import re
import gc
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sacrebleu.metrics import BLEU, CHRF
from tqdm.auto import tqdm

MODEL1_PATH = "/content/model1/"
MODEL2_PATH = "/content/model2/model/"
MODEL3_PATH = "/content/model3/google_madlad400_3b_mt_finetuned_corrected/"

TRAIN_CSV  = "/content/train.csv"
PREFIX     = "<2en> "
MAX_LENGTH = 576

# =============================================================================
# PREPROCESSING
# =============================================================================
_TO_SUBSCRIPT   = str.maketrans("0123456789", "₀₁₂₃₄₅₆₇₈₉")
_INDEX_DIGIT_RE = re.compile(r'(?<=[a-zA-ZÀ-ÿšŠṣṢṭṬẖḤ])(\d)')

def preprocess_input(text):
    if pd.isna(text):
        return ""
    text = str(text).strip()
    text = text.replace('<big_gap>', 'xxxxxx').replace('<gap>', 'x')
    text = re.sub(r'\{([^}]+)\}', r'(\1)', text)
    text = _INDEX_DIGIT_RE.sub(lambda m: m.group(0).translate(_TO_SUBSCRIPT), text)
    text = text.replace('H', 'Ḫ').replace('h', 'ḫ')
    return text

# =============================================================================
# POSTPROCESSING
# =============================================================================
_GAP = '\x02G\x02'
_BIG = '\x02B\x02'

_SLASH_FRAC_MAP = {
    ('1','2'):'½', ('1','3'):'⅓', ('2','3'):'⅔',
    ('1','4'):'¼', ('3','4'):'¾', ('1','6'):'⅙',
    ('5','6'):'⅚', ('1','8'):'⅛', ('3','8'):'⅜',
    ('5','8'):'⅝', ('7','8'):'⅞',
}
_SLASH_FRAC_RE = re.compile(r'(\d+)\s*/\s*(\d+)')

def _convert_slash_fracs(text):
    return _SLASH_FRAC_RE.sub(
        lambda m: _SLASH_FRAC_MAP.get((m.group(1), m.group(2)), m.group(0)), text)

_ROMAN = {
    'XII':'12', 'VIII':'8', 'VII':'7', 'VI':'6',
    'XI':'11',  'IX':'9',  'IV':'4',
    'III':'3',  'II':'2',  'X':'10',
    'I':'1',    'V':'5',
}
_ROMAN_RE = re.compile(
    r'\b[Mm]onth\s+(' + '|'.join(sorted(_ROMAN, key=len, reverse=True)) + r')\b')

_X_PLUS_RE = re.compile(r'\bx\s*\+\s*', re.IGNORECASE)
_X_RE      = re.compile(r'(?<!\w)x(?!\w)')

def _resolve_brackets(text):
    def _repl(m):
        inner = m.group(1).strip()
        if not inner:
            return _GAP
        if (re.fullmatch(r'[\.x\s\+\d]*', inner, re.IGNORECASE) and
                not re.search(r'[a-zA-ZÀ-ÿšŠṣṢ]', inner)):
            return _GAP
        cleaned = re.sub(r'(?<!\w)x(?!\w)|\.{3,}', ' ' + _GAP + ' ', inner)
        return re.sub(r'\s+', ' ', cleaned).strip() or _GAP
    return re.sub(r'\[([^\]]*)\]', _repl, text)

def postprocess_output(text):
    if not isinstance(text, str):
        return str(text) if text else "broken text"
    if not text.strip():
        return "broken text"
    text = text.replace('<big_gap>', _BIG).replace('<gap>', _GAP)
    text = _X_PLUS_RE.sub(_GAP + ' ', text)
    text = re.sub(r'\.{2,}', _GAP, text)
    text = re.sub(r'…',      _GAP, text)
    text = re.sub(r'\bxxxx+\b', _GAP, text)
    text = text.replace(_GAP, '<gap>').replace(_BIG, '<big_gap>')
    text = _resolve_brackets(text)
    text = text.replace('<big_gap>', _BIG).replace('<gap>', _GAP)
    text = _X_RE.sub(_GAP, text)
    for ch in ('"', '\u201c', '\u201d', '[', ']', '+', '(', ')'):
        text = text.replace(ch, '')
    text = re.sub(r"'s\b",      's', text)
    text = re.sub(r'\u2019s\b', 's', text)
    text = _ROMAN_RE.sub(lambda m: 'Month ' + _ROMAN[m.group(1).upper()], text)
    text = _convert_slash_fracs(text)
    text = text.replace(_GAP, '<gap>').replace(_BIG, '<gap>')
    text = re.sub(r'(<gap>\s*){2,}', '<gap> ', text)
    text = re.sub(r'\{[^}]+\}', '', text)
    text = text.replace('ḫ', 'h').replace('Ḫ', 'H')
    for sub, asc in zip('₀₁₂₃₄₅₆₇₈₉', '0123456789'):
        text = text.replace(sub, asc)
    text = re.sub(r'(?<=\s)-textiles\b', 'kutānu-textiles', text)
    text = re.sub(r'(?<=\s)-textile\b',  'kutānum-textile',  text)
    text = re.sub(r"(?<=\w)'(?=\w)", '', text)
    text = text.replace("'", '')
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'^[-–]+\s*', '', text)
    text = re.sub(r'\s*[-–]+$', '', text)
    text = re.sub(r'\s*<gap>\s*', ' <gap> ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text if text else "broken text"

# =============================================================================
# LOAD & PREPARE VALIDATION DATA FROM TRAIN.CSV
# =============================================================================
print(f"Reading {TRAIN_CSV}...")
train_df = pd.read_csv(TRAIN_CSV)

# Use random 350 rows as validation set
val_df = train_df.sample(n=350, random_state=42).reset_index(drop=True)

val_sources    = val_df["transliteration"].apply(preprocess_input).tolist()
val_references = val_df["translation"].tolist()  # ← adjust column name if different

print(f"Validation set: {len(val_df)} samples")

# =============================================================================
# MEASURE ALL MODELS
# =============================================================================
def measure_all_models(val_sources, val_references, model_paths):
    bleu_metric = BLEU(effective_order=True)
    chrf_metric = CHRF(word_order=2)  # chrF++

    results = {}

    for path in model_paths:
        print(f"\nEvaluating: {path}")
        tok = AutoTokenizer.from_pretrained(path)
        mdl = AutoModelForSeq2SeqLM.from_pretrained(
                path, device_map="auto",
                torch_dtype=torch.float32,
                low_cpu_mem_usage=True).eval()

        preds = []
        for text in tqdm(val_sources, desc="Translating"):
            input_text = PREFIX + str(text).strip()
            if not input_text.strip():
                preds.append("broken text")
                continue
            inputs = tok(
                input_text, return_tensors="pt",
                truncation=True, max_length=MAX_LENGTH
            ).to(mdl.device)

            with torch.inference_mode():
                out = mdl.generate(
                    **inputs, max_length=MAX_LENGTH,
                    num_beams=4, early_stopping=True, length_penalty=1.09)

            raw = tok.decode(out[0], skip_special_tokens=True)
            preds.append(postprocess_output(raw))  # ← apply your postprocessing

        b = bleu_metric.corpus_score(preds, [val_references]).score
        c = chrf_metric.corpus_score(preds, [val_references]).score
        g = (b * c) ** 0.5

        print(f"  BLEU    = {b:.2f}")
        print(f"  chrF++  = {c:.2f}")
        print(f"  GeoMean = {g:.2f}  ← competition metric")

        results[path] = {"bleu": b, "chrf": c, "geo": g}

        del mdl
        gc.collect()
        torch.cuda.empty_cache()

    return results

# =============================================================================
# RUN EVALUATION
# =============================================================================
model_paths = [MODEL1_PATH, MODEL2_PATH, MODEL3_PATH]
scores = measure_all_models(val_sources, val_references, model_paths)

# =============================================================================
# PRINT FINAL SUMMARY
# =============================================================================
print(f"\n{'='*60}")
print("FINAL SCORES SUMMARY")
print(f"{'='*60}")
for path, s in scores.items():
    print(f"{path.split('/')[-1]}")
    print(f"  BLEU={s['bleu']:.2f} | chrF++={s['chrf']:.2f} | GeoMean={s['geo']:.2f}")

print(f"\n→ Copy these GeoMean values into your merging script as weights!")
print(f"{'='*60}")

Reading /content/train.csv...
Validation set: 350 samples

Evaluating: /content/model1/


The tokenizer you are loading from '/content/model1/' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.


Translating:   0%|          | 0/350 [00:00<?, ?it/s]

  BLEU    = 36.81
  chrF++  = 57.98
  GeoMean = 46.20  ← competition metric

Evaluating: /content/model2/model/


... (6 outputs left out)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.


Translating:   0%|          | 0/350 [00:00<?, ?it/s]

  BLEU    = 34.68
  chrF++  = 55.62
  GeoMean = 43.92  ← competition metric

FINAL SCORES SUMMARY

  BLEU=36.81 | chrF++=57.98 | GeoMean=46.20

  BLEU=41.39 | chrF++=62.12 | GeoMean=50.71

  BLEU=34.68 | chrF++=55.62 | GeoMean=43.92

→ Copy these GeoMean values into your merging script as weights!


# based on above we model 2 better chrff score so that's why for tokenization we consider model 2

#merging initiated

In [ ]:
!rm -rf /content/merged_model/

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory


In [ ]:
import re
import gc
import json
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sacrebleu.metrics import BLEU, CHRF
from tqdm.auto import tqdm
import os

MODEL1_PATH = "/content/model1/"
MODEL2_PATH = "/content/model2/model/"
MODEL3_PATH = "/content/model3/google_madlad400_3b_mt_finetuned_corrected/"
TOKENIZER_PATH = MODEL2_PATH  # Model 2 — trained on full dataset, broadest vocab coverage

TRAIN_CSV  = "/content/train.csv"
PREFIX     = "<2en> "
MAX_LENGTH = 576
BATCH_SIZE = 16
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

# =============================================================================
# SCORES — Kaggle leaderboard BLEU + local 350-sample chrF++
# =============================================================================
bleu1, chrf1 = 35.3, 57.98
bleu2, chrf2 = 35.1, 62.12
bleu3, chrf3 = 31.6, 55.62

def geometric_mean(bleu, chrf):
    return (bleu * chrf) ** 0.5

geo1 = geometric_mean(bleu1, chrf1)
geo2 = geometric_mean(bleu2, chrf2)
geo3 = geometric_mean(bleu3, chrf3)

total_geo = geo1 + geo2 + geo3
w1 = geo1 / total_geo
w2 = geo2 / total_geo
w3 = geo3 / total_geo

print(f"Geometric means  → Model1: {geo1:.2f} | Model2: {geo2:.2f} | Model3: {geo3:.2f}")
print(f"Ensemble weights → w1: {w1:.4f} | w2: {w2:.4f} | w3: {w3:.4f}")

# =============================================================================
# LOAD MODELS & MERGE
# =============================================================================
print("\nLoading Model 1...")
model1 = AutoModelForSeq2SeqLM.from_pretrained(MODEL1_PATH)
sd1 = model1.state_dict()

print("Loading Model 2...")
model2 = AutoModelForSeq2SeqLM.from_pretrained(MODEL2_PATH)
sd2 = model2.state_dict()

print("Loading Model 3...")
model3 = AutoModelForSeq2SeqLM.from_pretrained(MODEL3_PATH)
sd3 = model3.state_dict()

print("Averaging weights...")
merged_sd = {}
for key in sd1:
    merged_sd[key] = w1 * sd1[key] + w2 * sd2[key] + w3 * sd3[key]

model = model1
model.load_state_dict(merged_sd)
model = model.to(DEVICE)
model.eval()

# Free unused models immediately
del model2, model3, sd1, sd2, sd3
gc.collect()
torch.cuda.empty_cache()

# =============================================================================
# TOKENIZER — Model 2 (trained on full dataset)
# =============================================================================
print(f"\nLoading tokenizer from {TOKENIZER_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)

# =============================================================================
# SAVE MERGED MODEL
# =============================================================================
SAVE_PATH = "/content/merged_model"
os.mkdir(SAVE_PATH)
print(f"\nSaving merged model to {SAVE_PATH}...")
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

# =============================================================================
# SAVE MERGE METADATA
# =============================================================================
metadata = {
    "strategy": "geometric_mean_weighted_averaging",
    "tokenizer": {
        "source_model": TOKENIZER_PATH,
        "reason": "Model 2 trained on full dataset — broadest vocabulary coverage"
    },
    "models": {
        "model1": {
            "path"   : MODEL1_PATH,
            "bleu"   : bleu1,
            "chrf"   : chrf1,
            "geo"    : round(geo1, 4),
            "weight" : round(w1, 4),
            "note"   : "Trained on subset of Model 2 data — best Kaggle BLEU"
        },
        "model2": {
            "path"   : MODEL2_PATH,
            "bleu"   : bleu2,
            "chrf"   : chrf2,
            "geo"    : round(geo2, 4),
            "weight" : round(w2, 4),
            "note"   : "Trained on full dataset — highest local GeoMean and chrF++"
        },
        "model3": {
            "path"   : MODEL3_PATH,
            "bleu"   : bleu3,
            "chrf"   : chrf3,
            "geo"    : round(geo3, 4),
            "weight" : round(w3, 4),
            "note"   : "Trained on subset of Model 2 data"
        },
    },
    "score_sources": {
        "bleu"  : "Kaggle leaderboard — true unseen test set",
        "chrf"  : "Local 350-sample validation from train.csv",
        "geo"   : "Geometric mean of above two scores"
    },
    "weights_sum_check": round(w1 + w2 + w3, 6)  # should always be 1.0
}

with open(f"{SAVE_PATH}/merge_config.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\n{'='*60}")
print(f"✓ Merged model saved to  : {SAVE_PATH}")
print(f"✓ Tokenizer source       : {TOKENIZER_PATH}")
print(f"✓ Weights sum            : {metadata['weights_sum_check']} (should be 1.0)")
print(f"✓ merge_config.json saved")
print(f"{'='*60}")

Geometric means  → Model1: 45.24 | Model2: 46.69 | Model3: 41.92
Ensemble weights → w1: 0.3380 | w2: 0.3488 | w3: 0.3132

Loading Model 1...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.


Loading Model 2...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.


... (2 outputs left out)

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.


Averaging weights...

Loading tokenizer from /content/model2/model/...


The tokenizer you are loading from '/content/model2/model/' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.



Saving merged model to /content/merged_model...

✓ Merged model saved to  : /content/merged_model
✓ Tokenizer source       : /content/model2/model/
✓ Weights sum            : 1.0 (should be 1.0)
✓ merge_config.json saved


In [ ]:
!rm -r /content/kaggle.json

rm: cannot remove '/content/kaggle.json': No such file or directory


In [ ]:
%%writefile /content/kaggle.json
{
"username":"ravikshdikola","key":"apply token here"
}


Writing /content/kaggle.json


In [ ]:
# import os
# os.chdir("/content")

In [ ]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
%cd /content/merged_newoldmix_version2/


/content/merged_newoldmix_version2


In [ ]:
%%writefile dataset-metadata.json
{
  "title": "madlab Akkadian retrained_1st_stage_newold_merged",
  "id": "ravikshdikola/madlab-akkadian-retrained-1st-newold-merged",
  "licenses": [
    {
      "name": "CC0-1.0"
    }
  ]
}


Writing dataset-metadata.json


In [ ]:
# %%writefile dataset-metadata.json
# {
#   "title": "madlab Akkadian retrained_1st_stage_merged_ver2",
#   "id": "ravikshdikola/madlab-akkadian-retrained-1st-stage-merged-ver2",
#   "licenses": [
#     {
#       "name": "CC0-1.0"
#     }
#   ]
# }


Writing dataset-metadata.json


In [ ]:
!kaggle datasets create -p .

Starting upload for file merge_config.json
100% 879/879 [00:00<00:00, 1.51kB/s]
Upload successful: merge_config.json (879B)
Starting upload for file generation_config.json
100% 122/122 [00:00<00:00, 217B/s]
Upload successful: generation_config.json (122B)
Starting upload for file tokenizer.json
100% 15.8M/15.8M [00:00<00:00, 21.6MB/s]
Upload successful: tokenizer.json (16MB)
Starting upload for file model-00002-of-00003.safetensors
100% 4.66G/4.66G [00:53<00:00, 92.7MB/s]
Upload successful: model-00002-of-00003.safetensors (5GB)
Starting upload for file model-00003-of-00003.safetensors
100% 1.66G/1.66G [00:20<00:00, 85.2MB/s]
Upload successful: model-00003-of-00003.safetensors (2GB)
Starting upload for file spiece.model
100% 4.22M/4.22M [00:00<00:00, 6.06MB/s]
Upload successful: spiece.model (4MB)
Starting upload for file config.json
100% 799/799 [00:00<00:00, 1.33kB/s]
Upload successful: config.json (799B)
Starting upload for file special_tokens_map.json
100% 414/414 [00:00<00:00, 718

2nd merging

In [ ]:
import kagglehub
import os

# ⚠️ Better practice: use secrets, but keeping your example intact
os.environ["KAGGLE_USERNAME"] = "ravikshdikola"
os.environ["KAGGLE_KEY"] = "apply token here"

# 👇 Force kagglehub cache to Colab filesystem
#os.environ["KAGGLEHUB_CACHE_DIR"] = "/content/kagglehub"

# # Download dataset
path = kagglehub.dataset_download("ravikshdikola/madlab-akkadian-retrained-1st-stage-merged")

print("Dataset downloaded to:", path)


100%|██████████| 10.2G/10.2G [01:26<00:00, 125MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/ravikshdikola/madlab-akkadian-retrained-1st-stage-merged/versions/1


In [ ]:
import kagglehub
import os

# ⚠️ Better practice: use secrets, but keeping your example intact
os.environ["KAGGLE_USERNAME"] = "ravikshdikola"
os.environ["KAGGLE_KEY"] = "apply token here"

# 👇 Force kagglehub cache to Colab filesystem
#os.environ["KAGGLEHUB_CACHE_DIR"] = "/content/kagglehub"

# # Download dataset
path = kagglehub.dataset_download("ravikshdikola/madlab-akkadian-retrained-1st-stage-newmix")

print("Dataset downloaded to:", path)


100%|██████████| 10.2G/10.2G [01:04<00:00, 170MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/ravikshdikola/madlab-akkadian-retrained-1st-stage-newmix/versions/1


In [ ]:
!cp -r /root/.cache/kagglehub/datasets/ravikshdikola/madlab-akkadian-retrained-1st-stage-merged/versions/1  /content/model1/

In [ ]:
!cp -r /root/.cache/kagglehub/datasets/ravikshdikola/madlab-akkadian-retrained-1st-stage-newmix/versions/1  /content/model2/

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
================================================================================
Multi-Model Akkadian→English Evaluation Pipeline  (v2 — gap-token aligned)
================================================================================
KEY FIX vs v1
─────────────────────────────────────────────────────────────────────────────
The training preprocess_input converts:
    xx+  →  <big_gap>
    x    →  <gap>

So the model's INPUT vocabulary contains <gap> / <big_gap> tokens, NOT bare x.

v1 inference did the OPPOSITE (converted <gap>→x, <big_gap>→xxxxxx before
feeding the model), which meant the model always saw out-of-distribution input.

v2 fix: after all other new→old conversions are done, we apply the SAME
gap normalisation as training:
    xxxxxx / xx+  →  <big_gap>
    standalone x  →  <gap>

This ensures the model sees exactly the same token space it was trained on.
─────────────────────────────────────────────────────────────────────────────
"""

import re
import gc
import unicodedata
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sacrebleu.metrics import BLEU, CHRF
from tqdm.auto import tqdm
from difflib import SequenceMatcher
from typing import List, Optional, Tuple

MODEL1_PATH = "/content/model1/"
MODEL2_PATH = "/content/model2/"

BATCH_SIZE = 16

TRAIN_CSV      = "/content/train.csv"
SENTENCES_CSV  = "/content/Sentences_Oare_FirstWord_LinNum.csv"
PREFIX         = "<2en> "
MAX_LENGTH     = 576
NUM_BEAMS      = 8
LENGTH_PENALTY = 1.5
MAX_DOC_CHARS  = 256          # docs longer than this are split into sentences
MIN_SENT_CHARS = 10           # discard sentence segments shorter than this
SEARCH_WINDOW  = 3            # ±words around expected first-word position

# ─────────────────────────────────────────────────────────────────────────────
# SHARED SENTINELS  (same as training pipeline)
# ─────────────────────────────────────────────────────────────────────────────

_GAP_SENTINEL    = '\x02GAP\x02'
_BIG_SENTINEL    = '\x02BIG\x02'
_HYPHEN_SENTINEL = '\x02HYP\x02'   # ← NEW in v2

_TO_SUBSCRIPT   = str.maketrans("0123456789", "₀₁₂₃₄₅₆₇₈₉")
_FROM_SUBSCRIPT = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")

_ACCENT_MAP = {
    'a2': 'á', 'a₂': 'á', 'A2': 'Á', 'A₂': 'Á',
    'e2': 'é', 'e₂': 'é', 'E2': 'É', 'E₂': 'É',
    'i2': 'í', 'i₂': 'í', 'I2': 'Í', 'I₂': 'Í',
    'u2': 'ú', 'u₂': 'ú', 'U2': 'Ú', 'U₂': 'Ú',
    'a3': 'à', 'a₃': 'à', 'A3': 'À', 'A₃': 'À',
    'e3': 'è', 'e₃': 'è', 'E3': 'È', 'E₃': 'È',
    'i3': 'ì', 'i₃': 'ì', 'I3': 'Ì', 'I₃': 'Ì',
    'u3': 'ù', 'u₃': 'ù', 'U3': 'Ù', 'U₃': 'Ù',
}

_KUBABBAR_RE = re.compile(r'KÙ\.B\.')
_DECIMAL_RE  = re.compile(r'(\d+\.\d{4})\d+')


# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 · PRE-PROCESSING  (new format → model-expected format)
# ─────────────────────────────────────────────────────────────────────────────
#
# CONVERSION TABLE (new → what model was trained on)
# ───────────────────────────────────────────────────────────────────────────
# new format              →  model-expected (training canonical form)
# ───────────────────────────────────────────────────────────────────────────
# {d}, {ki}, {m} …        →  (d), (ki), (m) …
# h / H                   →  ḫ / Ḫ   (outside determinatives)
# a2 / a₂, e2 / e₂ …     →  á, é …
# KÙ.B.                   →  KÙ.BABBAR
# 0.33333…                →  0.3333  (4 dp max)
# <big_gap>               →  <big_gap>  ← kept as gap tag (trained on this)
# <gap>                   →  <gap>      ← kept as gap tag (trained on this)
# xx+ bare runs           →  <big_gap>  (normalise any remaining bare x-runs)
# standalone bare x       →  <gap>      (normalise any remaining bare x)
# ───────────────────────────────────────────────────────────────────────────
#
# *** KEY CHANGE vs v1 ***
# v1 converted <gap>→x and <big_gap>→xxxxxx (WRONG — inverted the training mapping)
# v2 KEEPS <gap>/<big_gap> tags and normalises any remaining bare-x patterns
#    to these same tags, exactly mirroring training preprocess_input().
# ─────────────────────────────────────────────────────────────────────────────

def _convert_ascii_digits_to_accents(text: str) -> str:
    """Convert vowel+digit patterns to accented vowels (OLD training format)."""
    result = re.sub(
        r'([aAeEiIuU])([₂₃23])',
        lambda m: _ACCENT_MAP.get(m.group(1) + m.group(2),
                                   m.group(1) + m.group(2)),
        text
    )
    result = result.translate(_FROM_SUBSCRIPT)

    def _to_sub(m):
        char, digit = m.group(1), m.group(2)
        return char + digit.translate(_TO_SUBSCRIPT)

    result = re.sub(r'([a-zA-ZÀ-ÿšŠṣṢṭṬ])(\d)', _to_sub, result)
    return result


def _h_to_hbar(text: str) -> str:
    """h→ḫ, H→Ḫ everywhere EXCEPT inside {…} determinative blocks."""
    parts = re.split(r'(\{[^}]*\})', text)
    out = []
    for part in parts:
        if part.startswith('{') and part.endswith('}'):
            out.append(part)
        else:
            out.append(part.replace('H', 'Ḫ').replace('h', 'ḫ'))
    return ''.join(out)


def _normalise_gaps(text: str) -> str:
    """
    Normalise all gap representations → <gap> / <big_gap> tags.
    This mirrors training preprocess_input() exactly.

    *** NEW in v2 — replaces the old <big_gap>→xxxxxx / <gap>→x inversion ***

    Order matters:
      1. Protect existing tags (idempotency)
      2. Unicode ellipsis / multi-dot → <big_gap>
      3. xx+ runs → <big_gap>
      4. Standalone x → <gap>
      5. Restore tags
      6. Collapse adjacent gaps
    """
    # 1. Protect pre-existing tags
    text = text.replace('<big_gap>', _BIG_SENTINEL)
    text = text.replace('<gap>',     _GAP_SENTINEL)

    # 2. Unicode / multi-dot ellipsis → big_gap
    text = re.sub(r'……|…',  _BIG_SENTINEL, text)
    text = re.sub(r'\.{3,}', _BIG_SENTINEL, text)
    text = re.sub(r'(?:\.\s+){2,}\.', _BIG_SENTINEL, text)   # ". . ." style

    # 3. xx+ runs → big_gap
    text = re.sub(r'\bxx+\b', _BIG_SENTINEL, text, flags=re.IGNORECASE)

    # 4. Standalone x → gap  (not inside words like "ixtum")
    text = re.sub(r'(?<![A-Za-z\d])x(?![A-Za-z\d])', _GAP_SENTINEL, text)

    # 5. Restore sentinels as real tags
    text = text.replace(_BIG_SENTINEL, '<big_gap>')
    text = text.replace(_GAP_SENTINEL, '<gap>')

    # 6. Collapse adjacent gaps
    text = re.sub(r'(<big_gap>\s*){2,}', '<big_gap> ', text)
    text = re.sub(r'(<gap>\s*){2,}',     '<gap> ',     text)
    text = re.sub(r'<gap>\s*<gap>', '<big_gap>', text)

    return text


def preprocess_input(text) -> str:
    """
    Convert new-format transliteration → model-expected (old/training) format.

    new format              →  old / model-expected format
    ─────────────────────────────────────────────────────
    {d}, {ki}, {m} …        →  (d), (ki), (m) …
    h / H                   →  ḫ / Ḫ   (outside determinatives)
    a2 / a₂, e2 / e₂ …     →  á, é …
    KÙ.B.                   →  KÙ.BABBAR
    0.33333…                →  0.3333  (4 dp max)
    <big_gap> / xx+         →  <big_gap>   ← v2: keep as tag, not bare x
    <gap> / standalone x    →  <gap>       ← v2: keep as tag, not bare x
    ˹˺ / <<…>>              →  removed (scribal noise)
    """
    if pd.isna(text) or not isinstance(text, str):
        return ""
    text = text.strip()

    # 1. h → ḫ (before brace conversion so determinative interior is safe)
    text = _h_to_hbar(text)

    # 2. Determinatives: {X} → (X)
    text = re.sub(r'\{([^}]+)\}', r'(\1)', text)

    # 3. Digit subscripts / accents
    text = _convert_ascii_digits_to_accents(text)

    # 4. KÙ.B. expansion
    text = _KUBABBAR_RE.sub('KÙ.BABBAR', text)

    # 5. Decimal precision
    text = _DECIMAL_RE.sub(r'\1', text)

    # 6. Remove scribal noise characters
    text = text.replace('˹', '').replace('˺', '')
    text = re.sub(r'<<[^>]*>>', '', text)

    # 7. *** KEY STEP (v2) *** — normalise all gap representations → <gap>/<big_gap>
    #    Replaces old v1 logic that did the INVERSE conversion (wrong direction).
    text = _normalise_gaps(text)

    # 8. Normalise whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 · SENTENCE SPLITTING  (long docs → sentence-sized chunks)
# ─────────────────────────────────────────────────────────────────────────────

def _normalize_word(w: str) -> str:
    """Lowercase + strip punctuation for fuzzy first-word matching."""
    w = w.lower().strip('.,;:?!()[]"\'{}-')
    # Remove accents for comparison robustness
    w = unicodedata.normalize('NFD', w)
    w = ''.join(c for c in w if unicodedata.category(c) != 'Mn')
    return w


def _fuzzy_match(w1: str, w2: str, threshold: float = 0.80) -> bool:
    if w1 == w2:
        return True
    if min(len(w1), len(w2)) == 0:
        return False
    if min(len(w1), len(w2)) / max(len(w1), len(w2)) < 0.5:
        return False
    return SequenceMatcher(None, w1, w2).ratio() >= threshold


def _find_first_word_pos(doc_words_norm: List[str],
                         first_word_norm: str,
                         expected_idx: int,
                         window: int = SEARCH_WINDOW) -> int:
    """Search ±window around expected_idx for first_word."""
    for offset in range(-window, window + 1):
        idx = expected_idx + offset
        if 0 <= idx < len(doc_words_norm):
            if _fuzzy_match(doc_words_norm[idx], first_word_norm):
                return idx
    return -1


def split_document_into_sentences(
    translit: str,
    sentence_meta: pd.DataFrame,
) -> Optional[List[Tuple[str, str]]]:
    """
    Cut a long transliteration into sentence segments using the Sentences CSV metadata.

    Returns list of (translit_segment, sentence_translation) or None if splitting fails.
    The returned transliteration segments are in RAW (un-preprocessed) form so that
    preprocess_input() can be applied to each one individually.
    """
    words = translit.split()
    if len(words) < 3:
        return None

    words_norm = [_normalize_word(w) for w in words]

    cuts = []
    for _, row in sentence_meta.iterrows():
        fw_spell    = str(row.get('first_word_spelling', '')).strip()
        fw_num      = row.get('first_word_number', 0)
        sent_transl = str(row.get('translation', '')).strip()

        if not fw_spell or not sent_transl or sent_transl == 'nan':
            continue
        try:
            fw_num = int(fw_num)
        except (ValueError, TypeError):
            continue
        if fw_num < 1:
            continue

        fw_norm = _normalize_word(fw_spell)
        pos = _find_first_word_pos(words_norm, fw_norm, fw_num - 1)
        if pos == -1:
            continue

        cuts.append({'start': pos, 'translation': sent_transl})

    if not cuts:
        return None

    cuts.sort(key=lambda c: c['start'])

    segments = []
    for i, cut in enumerate(cuts):
        s = cut['start']
        e = cuts[i + 1]['start'] if i + 1 < len(cuts) else len(words)
        seg_translit = ' '.join(words[s:e])
        if len(seg_translit) < MIN_SENT_CHARS:
            continue
        segments.append((seg_translit, cut['translation']))

    return segments if segments else None


# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 · POST-PROCESSING  (model output → clean evaluation format)
# ─────────────────────────────────────────────────────────────────────────────

_ROMAN_MONTHS = [
    (r'\bmonth\s+XII\b',  'month 12'),
    (r'\bmonth\s+XI\b',   'month 11'),
    (r'\bmonth\s+VIII\b', 'month 8'),
    (r'\bmonth\s+VII\b',  'month 7'),
    (r'\bmonth\s+VI\b',   'month 6'),
    (r'\bmonth\s+IX\b',   'month 9'),
    (r'\bmonth\s+IV\b',   'month 4'),
    (r'\bmonth\s+III\b',  'month 3'),
    (r'\bmonth\s+II\b',   'month 2'),
    (r'\bmonth\s+X\b',    'month 10'),
    (r'\bmonth\s+V\b',    'month 5'),
    (r'\bmonth\s+I\b',    'month 1'),
]

_DECIMAL_FRACTIONS = [
    (r'\b0\.333+\d*\b',  '⅓'),  (r'(?<!\d)\.333+\d*\b', '⅓'),
    (r'\b0\.666+\d*\b',  '⅔'),  (r'(?<!\d)\.666+\d*\b', '⅔'),
    (r'\b0\.5\b',        '½'),   (r'(?<!\d)\.5\b',       '½'),
    (r'\b0\.25\b',       '¼'),
    (r'\b0\.75\b',       '¾'),
    (r'\b0\.125\b',      '⅛'),
    (r'\b0\.375\b',      '⅜'),
    (r'\b0\.625\b',      '⅝'),
    (r'\b0\.875\b',      '⅞'),
    (r'\b0\.166+\d*\b',  '⅙'),
    (r'\b0\.833+\d*\b',  '⅚'),
]

_X_PLUS_RE  = re.compile(r'\bx\s*\+\s*', re.IGNORECASE)
_X_RE       = re.compile(r'(?<!\w)x(?!\w)', re.IGNORECASE)
_BREAK_RE   = re.compile(
    r'\(\s*(?:large\s+break|\d+\s+broken\s+lines?|break)\s*\)',
    re.IGNORECASE)

_FORBIDDEN_CHARS = str.maketrans('', '', '""\u201c\u201d[]+'
                                         '\u02be\u02bf')   # ʾ ʿ


def _resolve_brackets(text: str) -> str:
    """[...] / [x] → <gap>,  [word] → word (keep editorial supplements)."""
    def _repl(m):
        inner = m.group(1).strip()
        if not inner:
            return _GAP_SENTINEL
        if re.fullmatch(r'[\.\sx\+\d]*', inner, re.IGNORECASE) and \
                not re.search(r'[a-zA-Z]', inner.replace('x', '')):
            return _GAP_SENTINEL
        cleaned = re.sub(r'(?i)(?<!\w)x(?!\w)|\.{3,}',
                         ' ' + _GAP_SENTINEL + ' ', inner)
        return re.sub(r'\s+', ' ', cleaned).strip() or _GAP_SENTINEL
    return re.sub(r'\[([^\]]*)\]', _repl, text)


def _remove_phrase_repeats(text: str) -> str:
    """Remove consecutive duplicate phrases of 2–8 words."""
    words = text.split()
    if len(words) < 4:
        return text
    for phrase_len in range(8, 1, -1):
        i = 0
        result, changed = [], False
        while i < len(words):
            if i + phrase_len * 2 <= len(words):
                phrase      = words[i:i + phrase_len]
                next_phrase = words[i + phrase_len:i + phrase_len * 2]
                if phrase == next_phrase:
                    result.extend(phrase)
                    j = i + phrase_len
                    while j + phrase_len <= len(words) and \
                            words[j:j + phrase_len] == phrase:
                        j += phrase_len
                    i = j
                    changed = True
                    continue
            result.append(words[i])
            i += 1
        if changed:
            words = result
    return ' '.join(words)


def postprocess_output(text) -> str:
    """
    Convert model output (old training format) → clean evaluation format.

    x / ...                 →  <gap>
    [word]                  →  word   (editorial supplement kept)
    [x] / [...]             →  <gap>
    (break) / (large break) →  <gap>
    roman month  (I–XII)    →  month 1–12
    decimal fractions       →  Unicode fractions (½ ⅓ ⅔ …)
    ḫ / Ḫ                   →  h / H
    subscript digits        →  ASCII digits
    duplicate words/phrases →  deduplicated
    """
    if not isinstance(text, str) or not text.strip():
        return "broken text"

    # 0. Sentinel existing gap markers
    text = text.replace('<big_gap>', _BIG_SENTINEL).replace('<gap>', _GAP_SENTINEL)

    # 1. (break) / (large break) / (N broken lines) → gap
    text = _BREAK_RE.sub(_GAP_SENTINEL, text)

    # 2. x+Number → <gap> Number
    text = _X_PLUS_RE.sub(_GAP_SENTINEL + ' ', text)

    # 3. Ellipsis → gap
    text = re.sub(r'\.{2,}', _GAP_SENTINEL, text)
    text = re.sub(r'…',      _GAP_SENTINEL, text)
    text = re.sub(r'\bxxxx+\b', _GAP_SENTINEL, text, flags=re.IGNORECASE)

    # 4. Resolve brackets
    text = text.replace(_GAP_SENTINEL, '<gap>').replace(_BIG_SENTINEL, '<big_gap>')
    text = _resolve_brackets(text)
    text = text.replace('<big_gap>', _BIG_SENTINEL).replace('<gap>', _GAP_SENTINEL)

    # 5. Standalone x → gap
    text = _X_RE.sub(_GAP_SENTINEL, text)

    # 6. Roman month numbers
    for pattern, replacement in _ROMAN_MONTHS:
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)

    # 7. Strip forbidden chars
    text = text.translate(_FORBIDDEN_CHARS)
    text = re.sub(r"^'+|'+$", '', text)
    text = re.sub(r"(?<!\w)'(?!\w)", '', text)
    text = re.sub(r"^\u2019+|\u2019+$", '', text)
    text = re.sub(r"(?<!\w)\u2019(?!\w)", '', text)

    # 8. Collapse big_gap → gap
    text = text.replace(_GAP_SENTINEL, '<gap>').replace(_BIG_SENTINEL, '<gap>')

    # 9. Collapse adjacent gaps
    text = re.sub(r'(?<!\S)<gap>\s+<gap>(?!\S)', '<gap>', text)
    text = re.sub(r'(<gap>\s+){3,}', '<gap> ', text)

    # 10. ḫ/Ḫ → h/H
    text = text.replace('ḫ', 'h').replace('Ḫ', 'H')

    # 11. Subscript digits → ASCII
    text = text.translate(_FROM_SUBSCRIPT)

    # 12. Decimal fractions → Unicode symbols
    for pattern, symbol in _DECIMAL_FRACTIONS:
        text = re.sub(pattern, symbol, text)
    for pattern, symbol in _DECIMAL_FRACTIONS:
        int_pat = r'\b(\d+)' + pattern.lstrip(r'\b0')
        text = re.sub(int_pat, r'\1 ' + symbol, text)
    text = re.sub(r'(\d)([½¼¾⅓⅔⅙⅚⅛⅜⅝⅞])', r'\1 \2', text)

    # 13. Deduplication
    text = re.sub(r'\b(\w+)(?:\s+\1\b)+', r'\1', text)
    text = _remove_phrase_repeats(text)

    # 14. Gap spacing
    text = re.sub(r'\s*<gap>\s*', ' <gap> ', text)

    # 15. Final whitespace / edge cleanup
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'^[-–]+\s*', '', text)
    text = re.sub(r'\s*[-–]+$', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text if text else "broken text"


# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 · LOAD & PREPARE VALIDATION DATA FROM TRAIN.CSV
# ─────────────────────────────────────────────────────────────────────────────
print(f"Reading {TRAIN_CSV}...")
train_df = pd.read_csv(TRAIN_CSV)

# Use random 350 rows as validation set
val_df = train_df.sample(n=500, random_state=42).reset_index(drop=True)

# Load sentences metadata for splitting long docs
sentences_grouped = None
try:
    sentences_df     = pd.read_csv(SENTENCES_CSV)
    sentences_grouped = sentences_df.groupby('text_uuid')
    print(f"      Sentences CSV loaded: {len(sentences_df)} rows, "
          f"{sentences_df['text_uuid'].nunique()} unique docs")
except FileNotFoundError:
    print(f"      ⚠ Sentences CSV not found — sentence splitting disabled")

# Build (preprocessed_source, reference) pairs, splitting long docs
val_sources    = []
val_references = []

for _, row in val_df.iterrows():
    raw_translit = row["transliteration"]
    reference    = row["translation"]
    doc_id       = str(row.get("id", row.name))

    # Try sentence-level splitting for long documents
    if (len(str(raw_translit)) > MAX_DOC_CHARS
            and sentences_grouped is not None
            and doc_id in sentences_grouped.groups):

        meta         = sentences_grouped.get_group(doc_id).sort_values('first_word_number')
        split_result = split_document_into_sentences(str(raw_translit), meta)

        if split_result and len(split_result) > 1:
            # preprocess_input applied PER SEGMENT on raw text
            seg_sources = [preprocess_input(seg) for seg, _ in split_result]
            val_sources.append(seg_sources)          # list-of-segments
            val_references.append(reference)
            continue

    # Default: single segment
    val_sources.append([preprocess_input(raw_translit)])
    val_references.append(reference)

# ── Sanity check: verify gap tokens present in preprocessed input ─────────────
sample_gaps = sum(
    1 for segs in val_sources
    for seg in segs
    if '<gap>' in seg or '<big_gap>' in seg
)
print(f"Validation set: {len(val_df)} samples  "
      f"({sum(1 for s in val_sources if len(s) > 1)} split into sentence segments)")
print(f"Segments containing <gap>/<big_gap> tokens: {sample_gaps}")


# ─────────────────────────────────────────────────────────────────────────────
# STEP 5 · MEASURE ALL MODELS
# ─────────────────────────────────────────────────────────────────────────────

def _translate_batch(model, tokenizer, texts: List[str], device) -> List[str]:
    """Tokenize, generate, and decode a batch of already-preprocessed strings."""
    inputs = tokenizer(
        [PREFIX + t for t in texts],
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True,
    ).to(device)

    with torch.inference_mode():
        tokens = model.generate(
            **inputs,
            max_length=MAX_LENGTH,
            num_beams=NUM_BEAMS,
            length_penalty=LENGTH_PENALTY,
            early_stopping=True,
        )

    return tokenizer.batch_decode(tokens, skip_special_tokens=True)


def measure_all_models(val_sources, val_references, model_paths):
    bleu_metric = BLEU(effective_order=True)
    chrf_metric = CHRF(word_order=2)  # chrF++

    results = {}

    for path in model_paths:
        print(f"\nEvaluating: {path}")
        tok = AutoTokenizer.from_pretrained(path)
        mdl = AutoModelForSeq2SeqLM.from_pretrained(
                path, device_map="auto",
                torch_dtype=torch.float16,
                low_cpu_mem_usage=True).eval()
        device = next(mdl.parameters()).device

        # ── Flatten all segments into one list for batched inference ─────────
        # flat: list of (doc_idx, seg_idx, preprocessed_text)
        flat = []
        for doc_idx, segments in enumerate(val_sources):
            for seg_idx, text in enumerate(segments):
                flat.append((doc_idx, seg_idx, str(text).strip() or "x"))

        raw_outputs = {}   # (doc_idx, seg_idx) → raw model string

        for start in tqdm(range(0, len(flat), BATCH_SIZE), desc="Translating"):
            batch   = flat[start : start + BATCH_SIZE]
            texts   = [item[2] for item in batch]
            decoded = _translate_batch(mdl, tok, texts, device)
            for (doc_idx, seg_idx, _), raw in zip(batch, decoded):
                raw_outputs[(doc_idx, seg_idx)] = raw
            # Periodic VRAM cleanup
            if start % (BATCH_SIZE * 20) == 0:
                torch.cuda.empty_cache()

        # ── Reassemble segments → full doc predictions ────────────────────────
        preds = []
        for doc_idx, segments in enumerate(val_sources):
            seg_translations = [
                postprocess_output(raw_outputs.get((doc_idx, seg_idx), ""))
                for seg_idx in range(len(segments))
            ]
            full = ' '.join(seg_translations)
            full = _remove_phrase_repeats(full)
            full = re.sub(r'\s+', ' ', full).strip()
            preds.append(full if full else "broken text")

        b = bleu_metric.corpus_score(preds, [val_references]).score
        c = chrf_metric.corpus_score(preds, [val_references]).score
        g = (b * c) ** 0.5

        print(f"  BLEU    = {b:.2f}")
        print(f"  chrF++  = {c:.2f}")
        print(f"  GeoMean = {g:.2f}  ← competition metric")

        results[path] = {"bleu": b, "chrf": c, "geo": g}

        del mdl
        gc.collect()
        torch.cuda.empty_cache()

    return results


# ─────────────────────────────────────────────────────────────────────────────
# STEP 6 · RUN EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
model_paths = [MODEL1_PATH, MODEL2_PATH]
scores = measure_all_models(val_sources, val_references, model_paths)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 7 · PRINT FINAL SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("FINAL SCORES SUMMARY")
print(f"{'='*60}")
for path, s in scores.items():
    print(f"{path.split('/')[-1]}")
    print(f"  BLEU={s['bleu']:.2f} | chrF++={s['chrf']:.2f} | GeoMean={s['geo']:.2f}")

print(f"\n→ Copy these GeoMean values into your merging script as weights!")
print(f"{'='*60}")

Reading /content/train.csv...
      Sentences CSV loaded: 9782 rows, 1700 unique docs
Validation set: 500 samples  (0 split into sentence segments)
Segments containing <gap>/<big_gap> tokens: 225

Evaluating: /content/model1/


The tokenizer you are loading from '/content/model1/' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.


Translating:   0%|          | 0/32 [00:00<?, ?it/s]

  BLEU    = 38.74
  chrF++  = 59.32
  GeoMean = 47.94  ← competition metric

Evaluating: /content/model2/


The tokenizer you are loading from '/content/model2/' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.


Translating:   0%|          | 0/32 [00:00<?, ?it/s]

  BLEU    = 48.85
  chrF++  = 66.27
  GeoMean = 56.90  ← competition metric

FINAL SCORES SUMMARY

  BLEU=38.74 | chrF++=59.32 | GeoMean=47.94

  BLEU=48.85 | chrF++=66.27 | GeoMean=56.90

→ Copy these GeoMean values into your merging script as weights!


#checking unknown token would be good option or not

# merging of 2nd version

In [ ]:
import re
import gc
import json
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sacrebleu.metrics import BLEU, CHRF
from tqdm.auto import tqdm
import os

MODEL1_PATH = "/content/model1/"
MODEL2_PATH = "/content/model2/"

TOKENIZER_PATH = MODEL2_PATH  # Model 3 — trained on full dataset, broadest vocab coverage

TRAIN_CSV  = "/content/train.csv"
PREFIX     = "<2en> "
MAX_LENGTH = 576
BATCH_SIZE = 16
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

# =============================================================================
# SCORES — Kaggle leaderboard BLEU + local 350-sample chrF++
# =============================================================================
bleu1, chrf1 = 35.8, 59.32
bleu2, chrf2 = 34.3, 66.27


def geometric_mean(bleu, chrf):
    return (bleu * chrf) ** 0.5

geo1 = geometric_mean(bleu1, chrf1)
geo2 = geometric_mean(bleu2, chrf2)


total_geo = geo1 + geo2
w1 = geo1 / total_geo
w2 = geo2 / total_geo


print(f"Geometric means  → Model1: {geo1:.2f} | Model2: {geo2:.2f}")
print(f"Ensemble weights → w1: {w1:.4f} | w2: {w2:.4f} ")

# =============================================================================
# LOAD MODELS & MERGE
# =============================================================================
print("\nLoading Model 1...")
model1 = AutoModelForSeq2SeqLM.from_pretrained(MODEL1_PATH)
sd1 = model1.state_dict()

print("Loading Model 2...")
model2 = AutoModelForSeq2SeqLM.from_pretrained(MODEL2_PATH)
sd2 = model2.state_dict()



print("Averaging weights...")
merged_sd = {}
for key in sd1:
    merged_sd[key] = w1 * sd1[key] + w2 * sd2[key]

model = model1
model.load_state_dict(merged_sd)
model = model.to(DEVICE)
model.eval()

# Free unused models immediately
del model2,  sd1, sd2,
gc.collect()
torch.cuda.empty_cache()

# =============================================================================
# TOKENIZER — Model 2 (trained on full dataset)
# =============================================================================
print(f"\nLoading tokenizer from {TOKENIZER_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)

# =============================================================================
# SAVE MERGED MODEL
# =============================================================================
SAVE_PATH = "/content/merged_newoldmix_version2"
os.mkdir(SAVE_PATH)
print(f"\nSaving merged model to {SAVE_PATH}...")
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

# =============================================================================
# SAVE MERGE METADATA
# =============================================================================
metadata = {
    "strategy": "geometric_mean_weighted_averaging",
    "tokenizer": {
        "source_model": TOKENIZER_PATH,
        "reason": "Model 2 trained on full dataset — broadest vocabulary coverage"
    },
    "models": {
        "model1": {
            "path"   : MODEL1_PATH,
            "bleu"   : bleu1,
            "chrf"   : chrf1,
            "geo"    : round(geo1, 4),
            "weight" : round(w1, 4),
            "note"   : "merged model which are trained on older dataset"
        },
        "model2": {
            "path"   : MODEL2_PATH,
            "bleu"   : bleu2,
            "chrf"   : chrf2,
            "geo"    : round(geo2, 4),
            "weight" : round(w2, 4),
            "note"   : "newly trained model trained on new format dataset with additional data "
        },

    },
    "score_sources": {
        "bleu"  : "Kaggle leaderboard — true unseen test set",
        "chrf"  : "Local 350-sample validation from train.csv",
        "geo"   : "Geometric mean of above two scores"
    },
    "weights_sum_check": round(w1 + w2 , 6)  # should always be 1.0
}

with open(f"{SAVE_PATH}/merge_config.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\n{'='*60}")
print(f"✓ Merged model saved to  : {SAVE_PATH}")
print(f"✓ Tokenizer source       : {TOKENIZER_PATH}")
print(f"✓ Weights sum            : {metadata['weights_sum_check']} (should be 1.0)")
print(f"✓ merge_config.json saved")
print(f"{'='*60}")

Geometric means  → Model1: 46.08 | Model2: 47.68
Ensemble weights → w1: 0.4915 | w2: 0.5085 

Loading Model 1...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.


Loading Model 2...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.


Averaging weights...

Loading tokenizer from /content/model2/...


The tokenizer you are loading from '/content/model2/' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.



Saving merged model to /content/merged_newoldmix_version2...

✓ Merged model saved to  : /content/merged_newoldmix_version2
✓ Tokenizer source       : /content/model2/
✓ Weights sum            : 1.0 (should be 1.0)
✓ merge_config.json saved


# merged model give me 35.80 on unseen set to improve it further doing hyperparamter tuning to imrpove score fruther

In [ ]:
import kagglehub
import os

# ⚠️ Better practice: use secrets, but keeping your example intact
os.environ["KAGGLE_USERNAME"] = "ravikshdikola"
os.environ["KAGGLE_KEY"] = "apply token here"

# 👇 Force kagglehub cache to Colab filesystem
#os.environ["KAGGLEHUB_CACHE_DIR"] = "/content/kagglehub"

# # Download dataset
path = kagglehub.dataset_download("ravikshdikola/madlab-akkadian-retrained-1st-newold-merged")

print("Dataset downloaded to:", path)


100%|██████████| 10.1G/10.1G [01:22<00:00, 132MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/ravikshdikola/madlab-akkadian-retrained-1st-newold-merged/versions/1


In [ ]:
!cp -r /root/.cache/kagglehub/datasets/ravikshdikola/madlab-akkadian-retrained-1st-newold-merged/versions/1  /content/model/

# so i have experimented with different the pipeline which gave me good i go with that for grid search

In [ ]:
import pandas as pd
tr = pd.read_csv('/content/train.csv')
tr = tr.sample(n=350, random_state=42).reset_index(drop=True)
tr.to_csv('/content/grid_search_sample.csv',index=False)

In [ ]:
"""
Grid search over generation hyperparameters for Akkadian→English MT model.
Evaluated on 350 random samples from train.csv using geometric mean of
BLEU and chrF++ (matching the Kaggle metric).

Key improvement: BATCHED inference with length-sorted bucketing for ~4-8x speedup.

Run on Kaggle:  python grid_search_batched.py
Results saved to: grid_search_results.csv
"""

import re, gc, json, itertools, time, math
import torch
import numpy as np
import pandas as pd
import sacrebleu
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm

# ─── PATHS ────────────────────────────────────────────────────────────────────
MODEL_PATH  = "/content/model/"
SAMPLE_CSV  = "/content/grid_search_sample.csv"
OUTPUT_CSV  = "/content/grid_search_results.csv"
PREFIX      = "<2en> "
MAX_INPUT   = 576

# ─── BATCH SIZE ───────────────────────────────────────────────────────────────
# Rule of thumb per GPU VRAM:
#   - T4  (16 GB) → BATCH_SIZE = 16
#   - P100(16 GB) → BATCH_SIZE = 16
#   - A100(40 GB) → BATCH_SIZE = 32
# Reduce if you hit OOM; the script will auto-halve on OOM and retry.
BATCH_SIZE = 32

# ─── PREPROCESSING (same as pipeline) ─────────────────────────────────────────
_TO_SUBSCRIPT   = str.maketrans("0123456789", "₀₁₂₃₄₅₆₇₈₉")
_INDEX_DIGIT_RE = re.compile(r'(?<=[a-zA-ZÀ-ÿšŠṣṢṭṬẖḤ])(\d)')
_KUBABBAR_RE    = re.compile(r'KÙ\.B\.')

def preprocess_input(text: str) -> str:
    if pd.isna(text): return ""
    text = str(text).strip()
    text = text.replace('<big_gap>', 'xxxxxx').replace('<gap>', 'x')
    def _h_to_hbar(t):
        parts = re.split(r'(\{[^}]*\})', t)
        return ''.join(
            p if (p.startswith('{') and p.endswith('}'))
            else p.replace('H', 'Ḫ').replace('h', 'ḫ')
            for p in parts)
    text = _h_to_hbar(text)
    text = re.sub(r'\{([^}]+)\}', r'(\1)', text)
    text = _INDEX_DIGIT_RE.sub(lambda m: m.group(0).translate(_TO_SUBSCRIPT), text)
    text = _KUBABBAR_RE.sub('KÙ.BABBAR', text)
    return text

# ─── POSTPROCESSING (same as pipeline) ────────────────────────────────────────
_GAP = '\x02G\x02'
_BIG = '\x02B\x02'
_X_PLUS_RE = re.compile(r'\bx\s*\+\s*', re.IGNORECASE)
_X_RE      = re.compile(r'(?<!\w)x(?!\w)')
_BREAK_RE  = re.compile(
    r'\(\s*(?:large\s+break|\d+\s+broken\s+lines?|break)\s*\)', re.IGNORECASE)

def _resolve_brackets(text):
    def _repl(m):
        inner = m.group(1).strip()
        if not inner: return _GAP
        if (re.fullmatch(r'[\.x\s\+\d]*', inner, re.IGNORECASE) and
                not re.search(r'[a-zA-ZÀ-ÿšŠṣṢ]', inner)):
            return _GAP
        cleaned = re.sub(r'(?<!\w)x(?!\w)|\.{3,}', ' ' + _GAP + ' ', inner)
        return re.sub(r'\s+', ' ', cleaned).strip() or _GAP
    return re.sub(r'\[([^\]]*)\]', _repl, text)

def postprocess_output(text):
    if not isinstance(text, str): return str(text) if text else "broken text"
    if not text.strip(): return "broken text"
    text = text.replace('<big_gap>', _BIG).replace('<gap>', _GAP)
    text = _BREAK_RE.sub(_GAP, text)
    text = _X_PLUS_RE.sub(_GAP + ' ', text)
    text = re.sub(r'\.{2,}', _GAP, text)
    text = re.sub(r'…', _GAP, text)
    text = re.sub(r'\bxxxx+\b', _GAP, text)
    text = text.replace(_GAP, '<gap>').replace(_BIG, '<big_gap>')
    text = _resolve_brackets(text)
    text = text.replace('<big_gap>', _BIG).replace('<gap>', _GAP)
    text = _X_RE.sub(_GAP, text)
    for ch in ('"', '\u201c', '\u201d', '[', ']', '+'): text = text.replace(ch, '')
    text = re.sub(r"^'+|'+$", '', text)
    text = re.sub(r"(?<!\w)'(?!\w)", '', text)
    text = re.sub(r"^\u2019+|\u2019+$", '', text)
    text = re.sub(r"(?<!\w)\u2019(?!\w)", '', text)
    text = text.replace(_GAP, '<gap>').replace(_BIG, '<gap>')
    text = re.sub(r'(?<!\S)<gap>\s+<gap>(?!\S)', '<gap>', text)
    text = re.sub(r'(<gap>\s+){3,}', '<gap> ', text)
    text = text.replace('ḫ', 'h').replace('Ḫ', 'H')
    for sub, asc in zip('₀₁₂₃₄₅₆₇₈₉', '0123456789'): text = text.replace(sub, asc)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'^[-–]+\s*', '', text)
    text = re.sub(r'\s*[-–]+$', '', text)
    text = re.sub(r'\s*<gap>\s*', ' <gap> ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text if text else "broken text"

# ─── METRIC ───────────────────────────────────────────────────────────────────
def compute_score(hypotheses: list, references: list) -> dict:
    """Geometric mean of corpus-level BLEU and chrF++ (micro-averaged)."""
    bleu_score = sacrebleu.corpus_bleu(hypotheses, [references]).score
    chrf_score = sacrebleu.corpus_chrf(
        hypotheses, [references], word_order=2  # chrF++ uses word_order=2
    ).score
    geo_mean = (bleu_score * chrf_score) ** 0.5
    return {"bleu": round(bleu_score, 4),
            "chrf": round(chrf_score, 4),
            "geo_mean": round(geo_mean, 4)}

# ─── LOAD DATA ────────────────────────────────────────────────────────────────
print("Loading sample data...")
df = pd.read_csv(SAMPLE_CSV)
df["transliteration"] = df["transliteration"].apply(preprocess_input)
sources    = df["transliteration"].tolist()
references = df["translation"].tolist()
n_samples  = len(df)
print(f"Loaded {n_samples} samples.")

# ─── LOAD MODEL ───────────────────────────────────────────────────────────────
print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
# Set padding token (needed for batched generation)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_PATH, device_map="auto", torch_dtype=torch.float16, low_cpu_mem_usage=True)
model.eval()
gc.collect()
torch.cuda.empty_cache()
print("Model loaded.\n")

# ─── PRE-TOKENIZE ALL INPUTS (done once, reused across all configs) ────────────
# Each sample encoded individually (no padding yet) → list of 1D tensors.
# We pad per-batch after length-sorting so padding waste is minimised.
print("Pre-tokenizing all inputs...")
encoded = [
    tokenizer(
        PREFIX + s.strip(),
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT,
    ).input_ids.squeeze(0)          # shape: (seq_len,)
    for s in tqdm(sources, desc="Tokenizing")
]
token_lengths = [t.shape[0] for t in encoded]

# ─── LENGTH-SORTED BUCKETS for minimal padding waste ──────────────────────────
# Sort indices by token length so sequences in the same batch are similar length.
# This is the single biggest speedup trick for batched MT inference.
sorted_indices = sorted(range(n_samples), key=lambda i: token_lengths[i])
print(f"Token length stats: min={min(token_lengths)}  "
      f"max={max(token_lengths)}  mean={sum(token_lengths)/len(token_lengths):.1f}")

# ─── BATCHED INFERENCE ────────────────────────────────────────────────────────
def translate_all_batched(cfg: dict, batch_size: int = BATCH_SIZE) -> list:
    """
    Translate all samples in sorted batches, then restore original order.
    Returns predictions in the ORIGINAL sample order (matching `references`).
    """
    results_sorted = []  # will hold (original_idx, hypothesis) pairs
    device = model.device

    gen_kwargs = dict(
        max_length     = cfg["max_length"],
        num_beams      = cfg["num_beams"],
        length_penalty = cfg["length_penalty"],
        early_stopping = True,
    )
    if cfg["no_repeat_ngram_size"] > 0:
        gen_kwargs["no_repeat_ngram_size"] = cfg["no_repeat_ngram_size"]

    # Process in sorted batches
    n_batches = math.ceil(n_samples / batch_size)
    with torch.inference_mode():
        for b in tqdm(range(n_batches), desc=f"  beams={cfg['num_beams']} lp={cfg['length_penalty']} no_rep={cfg['no_repeat_ngram_size']} maxl={cfg['max_length']}", leave=False):
            batch_orig_indices = sorted_indices[b * batch_size : (b + 1) * batch_size]
            batch_tensors      = [encoded[i] for i in batch_orig_indices]

            # Pad batch to max length in this mini-batch (left-pad for decoder models)
            # For encoder-decoder (T5/BART style) right-padding is standard.
            max_len = max(t.shape[0] for t in batch_tensors)
            padded  = torch.zeros(len(batch_tensors), max_len, dtype=torch.long)
            attn    = torch.zeros_like(padded)
            pad_id  = tokenizer.pad_token_id or 0

            for j, t in enumerate(batch_tensors):
                padded[j, :t.shape[0]] = t
                attn[j,   :t.shape[0]] = 1
            # Fill padding positions with pad_token_id
            padded[attn == 0] = pad_id

            padded = padded.to(device)
            attn   = attn.to(device)

            try:
                out = model.generate(
                    input_ids      = padded,
                    attention_mask = attn,
                    **gen_kwargs,
                )
            except torch.cuda.OutOfMemoryError:
                # OOM: fall back to half batch size recursively
                print(f"\n  ⚠  OOM on batch_size={batch_size}, retrying with {batch_size//2}")
                torch.cuda.empty_cache()
                # Re-process this batch sample-by-sample as fallback
                out_list = []
                for j in range(len(batch_tensors)):
                    single = padded[j:j+1][attn[j:j+1].bool().any(dim=1)]
                    single_ids = encoded[batch_orig_indices[j]].unsqueeze(0).to(device)
                    o = model.generate(single_ids, **gen_kwargs)
                    out_list.append(o[0])
                out = out_list  # list of 1D tensors
                del out_list

            # Decode each output
            if isinstance(out, list):
                decoded = [tokenizer.decode(o, skip_special_tokens=True) for o in out]
            else:
                decoded = tokenizer.batch_decode(out, skip_special_tokens=True)

            for orig_idx, raw in zip(batch_orig_indices, decoded):
                results_sorted.append((orig_idx, postprocess_output(raw)))

            del padded, attn, out
            torch.cuda.empty_cache()

    # Restore original order
    results_sorted.sort(key=lambda x: x[0])
    return [h for _, h in results_sorted]


# ─── GRID ─────────────────────────────────────────────────────────────────────
GRID = {
    "num_beams":             [4, 8, 12, 16],
    "length_penalty":        [0.8, 1.0, 1.18, 1.4, 1.6],
    "no_repeat_ngram_size":  [0, 2, 3, 4],
    "max_length":            [400, 576, 700],
}

keys   = list(GRID.keys())
combos = list(itertools.product(*[GRID[k] for k in keys]))
print(f"Total configurations to evaluate: {len(combos)}")
print(f"Batch size: {BATCH_SIZE}  |  Effective batches per config: "
      f"{math.ceil(n_samples / BATCH_SIZE)}\n")

# ─── RUN GRID SEARCH ──────────────────────────────────────────────────────────
records  = []
best_geo = -1.0
best_cfg = None

for i, vals in enumerate(tqdm(combos, desc="Grid Search", unit="cfg")):
    cfg = dict(zip(keys, vals))
    t0  = time.time()

    hyps   = translate_all_batched(cfg)
    scores = compute_score(hyps, references)
    elapsed = time.time() - t0

    row = {**cfg, **scores, "elapsed_s": round(elapsed, 1)}
    records.append(row)

    marker = ""
    if scores["geo_mean"] > best_geo:
        best_geo = scores["geo_mean"]
        best_cfg = cfg.copy()
        marker = "  ★ NEW BEST"

    print(f"[{i+1:>3}/{len(combos)}] geo={scores['geo_mean']:.4f}  "
          f"bleu={scores['bleu']:.2f}  chrf={scores['chrf']:.2f}  "
          f"beams={cfg['num_beams']}  lp={cfg['length_penalty']}  "
          f"no_rep={cfg['no_repeat_ngram_size']}  maxl={cfg['max_length']}  "
          f"({elapsed:.0f}s){marker}")

    # Incremental save — safe against session timeouts
    pd.DataFrame(records).sort_values("geo_mean", ascending=False).to_csv(OUTPUT_CSV, index=False)

    gc.collect()
    torch.cuda.empty_cache()

# ─── SUMMARY ──────────────────────────────────────────────────────────────────
results_df = pd.DataFrame(records).sort_values("geo_mean", ascending=False)
results_df.to_csv(OUTPUT_CSV, index=False)

print("\n" + "="*70)
print("GRID SEARCH COMPLETE")
print(f"Best geo_mean : {best_geo:.4f}")
print(f"Best config   : {json.dumps(best_cfg, indent=2)}")
print(f"Results saved : {OUTPUT_CSV}")
print("="*70)
print("\nTop-10 configurations:")
print(results_df.head(10).to_string(index=False))

Loading sample data...
Loaded 350 samples.
Loading model...


The tokenizer you are loading from '/content/model/' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.


Model loaded.

Pre-tokenizing all inputs...


Tokenizing:   0%|          | 0/350 [00:00<?, ?it/s]

... (195 outputs left out)

[ 97/240] geo=51.9972  bleu=43.05  chrf=62.81  beams=8  lp=1.4  no_rep=0  maxl=400  (323s)


  beams=8 lp=1.4 no_rep=0 maxl=576:   0%|          | 0/11 [00:00<?, ?it/s]

[ 98/240] geo=52.0921  bleu=43.19  chrf=62.83  beams=8  lp=1.4  no_rep=0  maxl=576  (477s)


  beams=8 lp=1.4 no_rep=0 maxl=700:   0%|          | 0/11 [00:00<?, ?it/s]